In [157]:
import numpy as np
import h5py
import os
import mne
import matplotlib.pyplot as plt
import pandas as pd
from typing import Tuple, List, Dict
import re
from pathlib import Path

In [2]:
things_metadata_dir = './THINGS_metadata'
preprocessed_dir = './processed'

## Average across stim reps

In [92]:
concepts_df = pd.read_csv(os.path.join(things_metadata_dir, 'concepts-metadata_things.tsv'), sep='\t') # for category avging when needed
image_metadata = np.load('./THINGS_metadata/image_metadata.npy', allow_pickle=True).item()
# train_img_concepts, train_img_concepts_THINGS, train_img_files (same for test)
print(f"Found {len(image_metadata['train_img_files'])} train images and {len(image_metadata['test_img_files'])} test images in metadata.")

Found 16540 train images and 200 test images in metadata.


In [134]:
def get_stim_img_and_category(stim_ids: np.ndarray, split: str) -> Tuple[List[str], List[str]]:
    """
    Given an array of stimulus IDs, return the corresponding image file paths and category labels.
    
    Args:
        stim_ids (np.ndarray): Array of stimulus IDs.
    Returns:
        img_files (List[str]): List of image file paths corresponding to the stimulus IDs.
        categories (List[str]): List of category labels corresponding to the stimulus IDs.
    """
    
    img_files = []
    categories = []

    all_img_paths = image_metadata[f'{split}_img_files']
    all_img_names = image_metadata[f'{split}_img_concepts']

    img_files = [all_img_paths[stim_id-1] for stim_id in stim_ids]
    
    # map stim_ids to their corresponding concepts and then to their categories
    for stim_id in stim_ids:
        concept = all_img_names[stim_id-1].split('_',1)[1] # split after number and _
        # replace _ with space
        concept = concept.replace('_', ' ')
        #remove any numbers from the concept
        concept = re.findall(r'[^\d]+', concept)[0].strip()
        if concept == "flip flop": concept = "flip-flop" # special case for flip flop which is not found in concepts_df
        
        if concept not in concepts_df['Word'].values:
            print(f"Warning: Concept '{concept}' not found in concepts_df. Assigning category as 'Unknown'.")
            return
        

        category = concepts_df.loc[concepts_df['Word'] == concept, 'Top-down Category (WordNet)'].values
        categories.append(category[0])
    
    return img_files, categories
    

In [5]:
def load_eeg_preprocessed(subject: int, config: int, split: str) -> Dict[str, np.ndarray]:
    eeg_data_path = os.path.join(preprocessed_dir, f'config_{config}', f'sub-{subject:02d}_{split}_epochs.h5')
    with h5py.File(eeg_data_path, 'r') as f:
        eeg      = f['eeg'][:]                              # loads fully into RAM
        stim_ids = f['stim_ids'][:]
        times    = f['times'][:]
        ch_names = [c.decode() for c in f['ch_names'][:]]  # bytes -> str
        meta     = dict(f.attrs)
    
    print(f"Loaded sub-{subject:02d} {split}:")
    print(f"  EEG      : {eeg.shape}  (epochs, channels, timepoints)")
    print(f"  Stim IDs : {stim_ids.shape}  unique={len(np.unique(stim_ids))}")
    print(f"  Times    : {times[0]*1000:.0f} to {times[-1]*1000:.0f} ms  "
          f"@ {1/(times[1]-times[0]):.0f} Hz")
    print(f"  Channels : {len(ch_names)}")
    
    return {'eeg': eeg, 'stim_ids': stim_ids, 'times': times, 'ch_names': ch_names, 'meta': meta}

In [6]:
def average_across_reps(eeg_data: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    eeg = eeg_data['eeg']
    stim_ids = eeg_data['stim_ids']
    
    unique_ids = np.unique(stim_ids)  

    erps = np.stack([
        eeg[stim_ids == sid].mean(axis=0)
        for sid in unique_ids
    ]).astype(np.float32)

    # Sanity check
    reps_per_stim = np.array([np.sum(stim_ids == sid) for sid in unique_ids])
    print(f"  Averaged: {len(eeg)} epochs → {len(unique_ids)} ERPs")
    print(f"  Reps per stimulus: min={reps_per_stim.min()}  "
          f"max={reps_per_stim.max()}  "
          f"mean={reps_per_stim.mean():.2f}")
    
    return erps, unique_ids

In [ ]:
# load preprocessed eeg data
subject = 1
config = 2
split = 'train'

eeg_data = load_eeg_preprocessed(subject, config, split)
stim_erps, stim_ids = average_across_reps(eeg_data)


Loaded sub-01 train:
  EEG      : (63673, 63, 251)  (epochs, channels, timepoints)
  Stim IDs : (63673,)  unique=16540
  Times    : -200 to 800 ms  @ 250 Hz
  Channels : 63
  Averaged: 63673 epochs → 16540 ERPs
  Reps per stimulus: min=1  max=6  mean=3.85


In [140]:
stim_erps.shape

(16540, 63, 251)

In [135]:
stim_img_files, stim_categories = get_stim_img_and_category(stim_ids, split)

In [148]:
ERP_component_windows = {
    "C1": {"spatial": [ 'Pz', 'POz', 'Oz'],  # posterior midline
           "temporal": (40, 100)},  
    "P1": {"spatial": ['PO7', 'PO3', 'O1', 'PO4', 'PO8', 'O2'],  # lateral occipital
           "temporal": (60, 130)},  
    "N1": {"spatial": ['PO7', 'PO3', 'POz', 'PO4', 'PO8'],  # parietal-occipital
           "temporal": (100, 200)},
    "N170": {"spatial": ['TP7', 'P7', 'PO7', 'PO8', 'P8', 'TP8'],  # inferior occipito-temporal
             "temporal": (110, 200)},
    "EPN": {"spatial":  ["PO7", "PO3", "PO4", "PO8"],               # lateral occipito-parietal
            "temporal": (200, 300)},
    "P2": {"spatial": ['CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6'], # centro-parietal
           "temporal": (150, 250)},
    "N2": {"spatial": ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6'], # fronto-central
           "temporal": (200, 350)},
    "P3b": {"spatial": ['CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6'], # centro-parietal
           "temporal": (300, 600)},
    "LPC": {"spatial": ['Pz', 'POz', 'P3', 'P4'], # parietal
           "temporal": (400, 800)},
    "N400": {"spatial": ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6'], # fronto-central
             "temporal": (300, 500)},
}

In [ ]:
def extract_erp_features(
    erps: np.ndarray,        # (n_stimuli, n_ch, n_t)
    times: np.ndarray,       # (n_t,) in seconds
    ch_names: list,
    component_windows: dict = ERP_component_windows,
) -> dict:
    """
    For each ERP component, extracts:
      scalar  : (n_stimuli,)          — mean amplitude over ROI channels × time window
      vector  : (n_stimuli, n_ch*n_t) — flattened spatiotemporal pattern
      ch_used : list[str]             — which channels were actually found
      t_used  : (n_t_roi,)            — timepoints used (ms)
    """
    times_ms = times * 1000
    results  = {}

    for comp, spec in component_windows.items():
        # --- channel mask ---
        valid_chs = [ch for ch in spec["spatial"] if ch in ch_names]
        missing   = [ch for ch in spec["spatial"] if ch not in ch_names]
        if missing:
            print(f"  [{comp}] channels not in montage (skipped): {missing}")
        if not valid_chs:
            print(f"  [{comp}] WARNING: no valid channels — skipping component")
            continue
        ch_idx = np.array([ch_names.index(ch) for ch in valid_chs])

        # --- temporal mask ---
        t0, t1 = spec["temporal"]
        t_mask = (times_ms >= t0) & (times_ms <= t1)
        if t_mask.sum() == 0:
            print(f"  [{comp}] WARNING: no timepoints in [{t0}, {t1}] ms — skipping")
            continue

        # roi_data : (n_stimuli, n_roi_ch, n_roi_t)
        roi_data = erps[:, ch_idx, :][:, :, t_mask]

        results[comp] = {
            "scalar":  roi_data.mean(axis=(1, 2)).astype(np.float32),   # (n_stimuli,)
            "vector":  roi_data.reshape(erps.shape[0], -1).astype(np.float32),
            "ch_used": valid_chs,
            "t_used":  times_ms[t_mask],
        }

        print(
            f"  {comp:6s} | "
            f"channels: {len(valid_chs):2d} {valid_chs}  |  "
            f"window: {t0}–{t1} ms ({t_mask.sum()} pts)  |  "
            f"scalar: {results[comp]['scalar'].shape}  "
            f"vector: {results[comp]['vector'].shape}"
        )

    return results


def save_erp_features(
    features:   dict,
    unique_ids: np.ndarray,
    subject_id: int,
    split:      str,
    output_dir: str,
):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    fpath = os.path.join(output_dir, f"sub-{subject_id:02d}_{split}_erp_features.h5")

    with h5py.File(fpath, "w") as f:
        f.create_dataset("unique_ids", data=unique_ids.astype(np.int32))
        f.attrs["subject_id"] = subject_id
        f.attrs["split"]      = split

        # per-component groups
        comp_grp = f.create_group("components")
        for comp, data in features.items():
            if comp == "__joint__":
                continue
            grp = comp_grp.create_group(comp)
            grp.create_dataset("scalar", data=data["scalar"], compression="gzip")
            grp.create_dataset("vector", data=data["vector"], compression="gzip")
            grp.create_dataset("t_used", data=data["t_used"])
            grp.attrs["ch_used"] = data["ch_used"]


    print(f"\n  Saved → {fpath}")
    return fpath


def load_erp_features(fpath: str) -> dict:
    """Load saved ERP features back into the same dict structure."""
    out = {"components": {}, "joint": {}}
    with h5py.File(fpath, "r") as f:
        out["unique_ids"] = f["unique_ids"][:]
        out["subject_id"] = f.attrs["subject_id"]
        out["split"]      = f.attrs["split"]

        for comp in f["components"]:
            grp = f["components"][comp]
            out["components"][comp] = {
                "scalar":  grp["scalar"][:],
                "vector":  grp["vector"][:],
                "t_used":  grp["t_used"][:],
                "ch_used": list(grp.attrs["ch_used"]),
            }

       
    return out


In [159]:
erp_features = extract_erp_features(stim_erps, eeg_data['times'], eeg_data['ch_names'])
save_erp_features(erp_features, stim_ids, subject, split, output_dir=f'./erp_processed/config{config}')

  C1     | channels:  3 ['Pz', 'POz', 'Oz']  |  window: 40–100 ms (16 pts)  |  scalar: (16540,)  vector: (16540, 48)
  P1     | channels:  6 ['PO7', 'PO3', 'O1', 'PO4', 'PO8', 'O2']  |  window: 60–130 ms (18 pts)  |  scalar: (16540,)  vector: (16540, 108)
  N1     | channels:  5 ['PO7', 'PO3', 'POz', 'PO4', 'PO8']  |  window: 100–200 ms (26 pts)  |  scalar: (16540,)  vector: (16540, 130)
  N170   | channels:  6 ['TP7', 'P7', 'PO7', 'PO8', 'P8', 'TP8']  |  window: 110–200 ms (23 pts)  |  scalar: (16540,)  vector: (16540, 138)
  EPN    | channels:  4 ['PO7', 'PO3', 'PO4', 'PO8']  |  window: 200–300 ms (26 pts)  |  scalar: (16540,)  vector: (16540, 104)
  P2     | channels:  7 ['CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6']  |  window: 150–250 ms (25 pts)  |  scalar: (16540,)  vector: (16540, 175)
  N2     | channels:  7 ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6']  |  window: 200–350 ms (38 pts)  |  scalar: (16540,)  vector: (16540, 266)
  P3b    | channels:  7 ['CP5', 'CP3', 'CP1

'./erp_processed/config2/sub-01_train_erp_features.h5'